# Tableau - LOD
- https://help.tableau.com/current/pro/desktop/en-us/calculations_calculatedfields_lod.htm

## LODs in Tableau

### LOD - FIXED
    - create as calculated field
    - Create Calculate field by name of
    - LOD Customer Sales and put this notation in text box {FIXED [Customer Name] : SUM([Sales])}

### LOD Region Sales
    - {FIXED [Region] : SUM([Sales]) }

### LOD Customer Sales
- {FIXED [Customer Name] : SUM([Sales])}

### LOD Customer First Order
- { FIXED [Customer Name] : MIN([Order Date]) }

### How to use in Sheets
- Case-1
    - Part - 1
        - Rows - Customer Name
        - Columns - Sales : Sum(Sales)
    - Part -2
        - Add Region to Rows, next to customer name
        - This will show Region wise sales of each customer
        - Add LOD Customer Sales to Columns
        - This will add another dimension to the sheet showing total sales by customer
        - Compare Region Sales of each customer with its total Sales in all regions

In [1]:
## Python - LOD

In [2]:
# Data
import numpy as np
import pandas as pd

In [5]:
url1 = 'https://raw.githubusercontent.com/dupadhyaya/piit/refs/heads/main/data/superstore_orders.csv'
print(url1)

https://raw.githubusercontent.com/dupadhyaya/piit/refs/heads/main/data/superstore_orders.csv


In [6]:
sales = pd.read_csv(url1)
sales.shape

(9994, 21)

In [7]:
sales.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,08/11/16,11/11/16,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,08/11/16,11/11/16,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,12/06/16,16/06/16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,11/10/15,18/10/15,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,11/10/15,18/10/15,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [32]:
sales.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')

In [67]:
# sales by each customer
custTotal = sales[['Customer Name','Sales']].groupby('Customer Name')['Sales']\
    .agg('sum').round(0).reset_index().rename(columns= {'Sales':'Total'})
custTotal.head()

,Customer Name,Total
0,Aaron Bergman,886.0
1,Aaron Hawkins,1745.0
2,Aaron Smayling,3051.0
3,Adam Bellavance,7756.0
4,Adam Hart,3250.0


In [69]:
# Sort sales in decreasing order
custTotal.sort_values(by='Total', ascending=False).head(10)

,Customer Name,Total
686,Sean Miller,25043.0
730,Tamara Chand,19052.0
622,Raymond Buch,15117.0
757,Tom Ashbrook,14596.0
6,Adrian Barton,14474.0
441,Ken Lonsdale,14175.0
671,Sanjit Chand,14142.0
334,Hunter Lopez,12873.0
672,Sanjit Engle,12209.0
156,Christopher Conant,12129.0


In [70]:
#group Customer Sales by Region
custRegion = sales[['Customer Name','Region','Sales']].groupby(['Customer Name','Region'])['Sales'] \
    .agg('sum').round(0).reset_index()
custRegion.head()

,Customer Name,Region,Sales
0,Aaron Bergman,Central,577.0
1,Aaron Bergman,West,310.0
2,Aaron Hawkins,East,330.0
3,Aaron Hawkins,South,86.0
4,Aaron Hawkins,West,1328.0


In [71]:
# join total Sales and region wise sales like LOD
custTotalRegion = pd.merge(custTotal, custRegion, on='Customer Name', how='inner')
custTotalRegion.shape

(2501, 4)

In [72]:
custTotalRegion.head()

,Customer Name,Total,Region,Sales
0,Aaron Bergman,886.0,Central,577.0
1,Aaron Bergman,886.0,West,310.0
2,Aaron Hawkins,1745.0,East,330.0
3,Aaron Hawkins,1745.0,South,86.0
4,Aaron Hawkins,1745.0,West,1328.0


In [73]:
custTotalRegion.groupby('Customer Name').head()

,Customer Name,Total,Region,Sales
0,Aaron Bergman,886.0,Central,577.0
1,Aaron Bergman,886.0,West,310.0
2,Aaron Hawkins,1745.0,East,330.0
3,Aaron Hawkins,1745.0,South,86.0
4,Aaron Hawkins,1745.0,West,1328.0
...,...,...,...,...
2496,Zuschuss Carroll,8026.0,South,1471.0
2497,Zuschuss Carroll,8026.0,West,2641.0
2498,Zuschuss Donatelli,1494.0,Central,331.0
2499,Zuschuss Donatelli,1494.0,South,857.0


In [74]:
## Index method
custTotal_idx = custTotal.set_index('Customer Name')
custRegion_idx = custRegion.set_index('Customer Name')
custRegion_idx.head()

,Region,Sales
Customer Name,,
Aaron Bergman,Central,577.0
Aaron Bergman,West,310.0
Aaron Hawkins,East,330.0
Aaron Hawkins,South,86.0
Aaron Hawkins,West,1328.0


In [75]:
custTotalRegion = custTotal_idx.join( custRegion_idx, how='inner')
custTotalRegion.head()

,Total,Region,Sales
Customer Name,,,
Aaron Bergman,886.0,Central,577.0
Aaron Bergman,886.0,West,310.0
Aaron Hawkins,1745.0,East,330.0
Aaron Hawkins,1745.0,South,86.0
Aaron Hawkins,1745.0,West,1328.0


- For pandas:
    * merge() → SQL-style joins on columns.
    * join() → Faster and cleaner when joining on indexes.
    * map() → Best when bringing a single aggregated value (like your customer total) into another DataFrame.

In [76]:
custRegion['Total'] = custRegion['Customer Name']\
    .map(custTotal.set_index('Customer Name')['Total'])
custRegion.sort_values(by='Total', ascending=False).head()

,Customer Name,Region,Sales,Total
2164,Sean Miller,Central,526.0,25043.0
2167,Sean Miller,West,10.0,25043.0
2166,Sean Miller,South,23669.0,25043.0
2165,Sean Miller,East,837.0,25043.0
2307,Tamara Chand,South,522.0,19052.0


In [77]:
#!pip install ipydatagrid
#from ipydatagrid import DataGrid
#DataGrid(custRegion)

In [78]:
#!pip install itables
from itables import show

In [79]:
show(custRegion)

Loading ITables v2.8.1 from the internet... (need help?)


In [80]:
GT(custRegion.head())

Customer Name,Region,Sales,Total
Aaron Bergman,Central,577.0,886.0
Aaron Bergman,West,310.0,886.0
Aaron Hawkins,East,330.0,1745.0
Aaron Hawkins,South,86.0,1745.0
Aaron Hawkins,West,1328.0,1745.0


In [84]:
## LOD - Minimum Order date
sales[['Customer Name', 'Order Date', 'Sales']].head()

,Customer Name,Order Date,Sales
0,Claire Gute,2016-08-11,261.9600
1,Claire Gute,2016-08-11,731.9400
2,Darrin Van Huff,2016-12-06,14.6200
3,Sean O'Donnell,2015-11-10,957.5775
4,Sean O'Donnell,2015-11-10,22.3680


In [85]:
sales['Order Date'] = pd.to_datetime(sales['Order Date'],format='%Y-%m-%d')
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9994 non-null   int64         
 1   Order ID       9994 non-null   object        
 2   Order Date     9994 non-null   datetime64[ns]
 3   Ship Date      9994 non-null   object        
 4   Ship Mode      9994 non-null   object        
 5   Customer ID    9994 non-null   object        
 6   Customer Name  9994 non-null   object        
 7   Segment        9994 non-null   object        
 8   Country        9994 non-null   object        
 9   City           9994 non-null   object        
 10  State          9994 non-null   object        
 11  Postal Code    9994 non-null   int64         
 12  Region         9994 non-null   object        
 13  Product ID     9994 non-null   object        
 14  Category       9994 non-null   object        
 15  Sub-Category   9994 n

In [88]:
custFirstOrder = (sales.groupby('Customer Name', as_index=False)['Order Date'].min()\
    .rename(columns={'Order Date': 'First Order Date'}))
custFirstOrder[['Customer Name', 'First Order Date']].sort_values(by='First Order Date')

,Customer Name,First Order Date
99,Bradley Drucker,2014-01-02
690,Shahid Collister,2014-01-03
202,Dave Brooks,2014-01-03
778,Vicky Freymann,2014-01-03
316,Hallie Redmond,2014-01-03
...,...,...
357,Jenna Caffey,2017-08-07
741,Theresa Coyne,2017-09-15
545,Mitch Gastineau,2017-10-04
584,Patricia Hirasaki,2017-10-21


In [93]:
custSummary = (sales.groupby('Customer Name', as_index=False).\
    agg(First_Order_Date =('Order Date','min'), Total_Sales = ('Sales','sum')))
custSummary.head()

,Customer Name,First_Order_Date,Total_Sales
0,Aaron Bergman,2014-02-18,886.156
1,Aaron Hawkins,2014-04-22,1744.700
2,Aaron Smayling,2014-07-27,3050.692
3,Adam Bellavance,2015-09-18,7755.620
4,Adam Hart,2014-11-16,3250.337


In [99]:
custSummary[['Customer Name', 'Total_Sales', 'First_Order_Date']].\
    sort_values(by=['Total_Sales', 'First_Order_Date'], ascending=[False,True])

,Customer Name,Total_Sales,First_Order_Date
686,Sean Miller,25043.050,2014-03-18
730,Tamara Chand,19052.218,2014-07-11
622,Raymond Buch,15117.339,2016-01-04
757,Tom Ashbrook,14595.620,2014-12-09
6,Adrian Barton,14473.571,2014-12-20
...,...,...,...
656,Roy Skaria,22.328,2016-07-06
545,Mitch Gastineau,16.739,2017-10-04
123,Carl Jackson,16.520,2016-12-30
455,Lela Donovan,5.304,2016-06-26


## EndHere